# 02 — Dense retrieval index over LEDGAR

Milestone **m2**: encode the LEDGAR training clauses with `all-MiniLM-L6-v2`, normalise embeddings, and retrieve the top-k clauses by cosine similarity. This notebook builds the index and presents 10 searches for manual inspection. It does not require a GPU; sentence-transformers will select an available device.

The clause bank is the **train split only**. The five pasted-clause examples below come from validation, so they are not their own matches in the index. Similarity is a relevance suggestion, not legal advice; inspect the returned text rather than treating the score as a relevance judgment.

## 1. Load LEDGAR and prepare the training clause bank

In [ ]:
from src.data import load_ledgar, to_pandas
from src.retrieval.dense import MODEL_NAME, build_dense_index
from src.utils import set_seed

set_seed(42)
dataset = load_ledgar()
train_df = to_pandas(dataset, "train")
validation_df = to_pandas(dataset, "validation")

print(f"Training clause bank: {len(train_df):,} clauses")
print(f"Validation clauses for query examples: {len(validation_df):,}")
print(f"Embedding model: {MODEL_NAME}")

## 2. Build or load the cached embedding index

The cached vectors are stored under `data/indexes/` (gitignored). The cache is reused only when both the model name and the ordered clause-bank contents match.

In [ ]:
retriever = build_dense_index(train_df, model_name=MODEL_NAME, batch_size=64)
cache_status = "loaded from cache" if retriever.cache_hit else "encoded and cached"
print(f"Index {cache_status}: {retriever.embeddings.shape[0]:,} clauses × {retriever.embeddings.shape[1]} dimensions")

## 3. Inspect 10 searches by hand

Five are lawyer-style questions and five are clauses held out from the training bank. Read the query and retrieved clause text together. For each search, note whether the results answer the question or are genuinely similar to the source clause; a shared broad topic alone is not enough. In particular, check whether numeric and jurisdiction constraints are actually preserved. These are qualitative checks, not a formal relevance-judgment set.

In [ ]:
lawyer_questions = [
    "Find clauses that allow either party to terminate the agreement for convenience with 30 days' written notice.",
    "Show clauses requiring payment of invoices within 30 days of receipt.",
    "Find agreements governed by New York law, excluding its conflict-of-law rules.",
    "Show clauses that cap liability at the fees paid in the preceding 12 months.",
    "Find confidentiality clauses that allow disclosure to affiliates and professional advisers.",
]

validation_examples = validation_df.sample(n=5, random_state=42)
queries = [(f"Lawyer question {i + 1}", query) for i, query in enumerate(lawyer_questions)]
queries += [
    (f"Validation clause idx={int(row.idx)} ({row.label_name})", row.text)
    for row in validation_examples.itertuples(index=False)
]

search_rows = []
for query_name, query_text in queries:
    for rank, result in enumerate(retriever.search(query_text, k=5), start=1):
        search_rows.append({
            "query": query_name,
            "rank": rank,
            "score": round(result["score"], 4),
            "idx": result["idx"],
            "label_name": result["label_name"],
            "text": result["text"],
        })

import pandas as pd
results_df = pd.DataFrame(search_rows)
for query_name in results_df["query"].drop_duplicates():
    print(f"\n{query_name}")
    display(results_df.loc[results_df["query"] == query_name, ["rank", "score", "label_name", "text"]].reset_index(drop=True))

## 4. Record the manual check

Use the table above to record which queries returned useful examples and where retrieval missed the intent. Do not use cosine similarity as a substitute for human relevance judgments. The broader lawyer-query evaluation, encoder comparison, and BM25 hybrid belong in notebook 05.